In [1]:
import sys
from pathlib import Path
import torch
from transformers import AutoProcessor, BitsAndBytesConfig
from transformers import Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

# Jupyter may start in either the repository root or the notebooks folder.
ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "Scripts" / "trainValSplit.py").is_file()),
    None,
)   
if ROOT is None:
    raise FileNotFoundError("Could not find the project root containing Scripts/trainValSplit.py")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from Scripts import dataUtils as dU
from Scripts import trainValSplit

MODEL_PATH = r"C:\models\HF\Qwen2.5-VL-3B-Instruct"
RUN_DIR = ROOT / "notebooks" / "checkpoints" / "qwen_manga_array_v2"

processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    min_pixels=128 * 28 * 28,
    max_pixels=256 * 28 * 28,
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    device_map={"": 0},
)


W1001 11:15:51.108000 23872 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

# MODEL EXPLORATION

In [ ]:
import torch.nn as nn

LinearLayerSeen = set()

for name, module in model.named_modules():

    if not isinstance(module, nn.Linear):
        continue

    leaf = name.split(".")[-1]
    
    key = (
        leaf,
        module.in_features,
        module.out_features,
    )

    if key in LinearLayerSeen:
        continue

    LinearLayerSeen.add(key)

    if ".visual." in name:
        branch = "VISION"
    elif ".language_model." in name:
        branch = "LANGUAGE"
    else:
        branch = "OTHER"

    print(
        f"{branch:10} | "
        f"{leaf:12} | "
        f"{module.in_features:6} -> {module.out_features:6} | "
        f"{name}"
    )

In [ ]:
from collections import defaultdict
import torch.nn as nn

info = defaultdict(lambda: {
    "count": 0,
    "shapes": set(),
    "examples": []
})

for name, module in model.named_modules():

    if not isinstance(module, nn.Linear):
        continue

    leaf = name.split(".")[-1]

    info[leaf]["count"] += 1
    info[leaf]["shapes"].add(
        (module.in_features, module.out_features)
    )

    if len(info[leaf]["examples"]) < 2:
        info[leaf]["examples"].append(name)


for leaf, x in info.items():

    print(f"\n{leaf}")
    print("  count :", x["count"])
    print("  shapes:", x["shapes"])

    for example in x["examples"]:
        print("  example:", example)

# LORA - TRAINING 

In [ ]:
# The base model and processor are loaded once in the setup cell above.
# Do not reload the model before evaluation; use the trained model in memory.

In [2]:
img_path = ROOT / "TaskData" / "development" / "images"
json_path = ROOT / "TaskData" / "development" / "labels.jsonl"

seqs = dU.load_labels(json_path).keys()
seqs = list(seqs)

# One label row maps to one naturally ordered image sequence.
X, y = trainValSplit.fit_seq(
    seqs, v_size=0.10, random_state=42
)

TASK_PROMPT = """
Treat the three manga images as one continuous story and process them in the order provided.

Read each manga page in Japanese reading order: panels from right to left and top to bottom, and text within each panel from right to left and top to bottom.

Transcribe every dialogue and narration text exactly as shown. Assign character labels by first appearance in the supplied image order: the first identifiable character is "character1", the next new character is "character2", and so on. Never restart character numbering on a new image. The same person must keep the same label across all three images. Use "UNKNOWN" only when the speaker cannot be identified and "NARRATION" for every narration box or caption.

Return only the target JSON array: one inner list per input image, in the same order as the images. Within each inner list, return objects containing only "speaker" and "text" in reading order. If an image has no dialogue or narration, return an empty list for that image.

Output valid JSON only, with no Markdown or explanation.
"""

prompt_builder = dU.MangaPromptBuilder(
    train_text=TASK_PROMPT
)


train_dataset = dU.MangaSequenceDataset(
    label_jsonl_path=json_path,
    image_root=img_path,
    sequence_ids=X
)

val_dataset = dU.MangaSequenceDataset(
    label_jsonl_path=json_path,
    image_root=img_path,
    sequence_ids=y,
)


train_collator = dU.MangaCollator(
    processor=processor,
    vision_info=process_vision_info,
    prompt_builder=prompt_builder,
    train_mode=True,
    # return_prompts=True
)

eval_collator = dU.MangaCollator(
    processor=processor,
    vision_info=process_vision_info,
    prompt_builder=prompt_builder,
    train_mode=False,
    return_metadata=True,
)

In [3]:
from peft import LoraConfig

# This experiment adapts only language-attention projections.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

The important fields are:
- r: the rank of the LoRA matrices. Higher rank = more trainable capacity, but more memory/parameters.
- lora_alpha: scaling factor for the LoRA update.
- target_modules: the actual layer names that LoRA should be inserted into.
- lora_dropout: dropout applied inside the LoRA branch during training.
- bias: whether biases should also be trainable.
- task_type: tells PEFT what kind of model/task you're adapting.

In [4]:
from Scripts.trainer import build_lora_model, build_training_args, build_trainer

lora_model = build_lora_model(model, lora_config)
lora_model.print_trainable_parameters()


trainable params: 37,152,768 || all params: 3,791,775,744 || trainable%: 0.9798


In [6]:
training_args = build_training_args(
    output_dir=str(RUN_DIR),
    batch_size=1,
    gradient_accumulation_steps=10,
    learning_rate=1e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    max_grad_norm=1.0,
    dataloader_num_workers=0,
    warmup_steps=7,
    lr_scheduler_type="cosine",
    load_best_model_at_end=True,
)

trainer = build_trainer(
    model=lora_model,
    train_dataset=train_dataset,
    data_collator=train_collator,
    training_args=training_args,
    eval_dataset=val_dataset,
)

trainer.train()

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,0.501737,0.504195
2,0.386036,0.477971
3,0.349882,0.471721


TrainOutput(global_step=138, training_loss=0.43901373856309533, metrics={'train_runtime': 7277.455, 'train_samples_per_second': 0.188, 'train_steps_per_second': 0.019, 'total_flos': 4.077250509511066e+16, 'train_loss': 0.43901373856309533, 'epoch': 3.0})

In [7]:
FINAL_DIR = RUN_DIR / "final"
trainer.save_model(str(FINAL_DIR))
processor.save_pretrained(str(FINAL_DIR))

['C:\\Users\\ESHAAN\\BYTE\\Applied-Task\\manga_vlm_crashcourse\\notebooks\\checkpoints\\qwen_manga_array\\final\\processor_config.json']

| Count | Per epoch | Entire run |
|---|---:|---:|
| Dataset examples processed | 72 | 216 |
| Images processed | 216 | 648 |
| Micro-batches / forward-backward passes | 72 | 216 |
| Optimizer-update steps | 8 | 24 |
| Validation examples | 8 | 24 |


There are two relevant kinds of training steps:
1. Micro-step
   One batch containing one manga sequence is passed through the model. Loss and gradients are calculated, but the weights are not necessarily updated.
2. Optimizer step / global step
   After gradients from 10 micro-steps have accumulated, the optimizer clips the global gradient norm to 1.0 and updates the LoRA weights once. Validation loss is then measured at the end of each epoch.
Therefore:


`1 full optimizer step = 10 sequences = 30 manga images` (the final step of an epoch may contain fewer sequences).

# EVALUATION

This generates JSON for every validation sequence, matches dialogues by text, and reports all evaluation metrics.

- `cer_valid_outputs`: transcription errors among schema-valid outputs (lower is better)
- `cer_including_failures_as_deletions`: conservative companion CER that treats invalid outputs as fully missing
- `dialogue_text_match_f1`: approximate dialogue/text matches; this is not spatial bubble detection
- `matched_dialogue_order_accuracy`: reading order conditional on matched dialogues
- `matched_speaker_accuracy`: speaker assignment conditional on matched dialogues, after aligning arbitrary character labels
- `b3_f1`: character-identity grouping consistency
- `identity_switch_rate`: cross-page identity changes (lower is better)
- `narration_f1`: narration detection
- `json_parse_rate` and `schema_compliance_rate`: output formatting
- `sequence_exact_match`: completely correct output after speaker-label alignment
- `literal_sequence_exact_match`: exact output without speaker-label alignment
- `generation_diagnostics`: token-limit and repetition-loop diagnostics

Every validation row is one naturally ordered manga sequence. All metrics are between 0 and 1 except CER, which can exceed 1.

Evaluate the freshly trained `lora_model` already in memory. Loading the base model a second time in this notebook can exhaust GPU memory or the Windows paging file.

The validation dataset, inference collator, and prompt were created once in the data-setup cell. The target remains metadata and is never included in the generation prompt.

In [8]:
from Scripts.evaluator import evaluate_model

val_metrics = evaluate_model(
    model=lora_model,
    processor=processor,
    dataset=val_dataset,
    collator=eval_collator,
    batch_size=1,
    max_new_tokens=1024,
    minimum_similarity=0.50,
    repetition_penalty=1.05,
    output_dir=ROOT / "outputs" / "qwen_manga_array_eval",
)
val_metrics

Evaluated 1/24 sequences
Evaluated 2/24 sequences
Evaluated 3/24 sequences
Evaluated 4/24 sequences
Evaluated 5/24 sequences
Evaluated 6/24 sequences
Evaluated 7/24 sequences
Evaluated 8/24 sequences
Evaluated 9/24 sequences
Evaluated 10/24 sequences
Evaluated 11/24 sequences
Evaluated 12/24 sequences
Evaluated 13/24 sequences
Evaluated 14/24 sequences
Evaluated 15/24 sequences
Evaluated 16/24 sequences
Evaluated 17/24 sequences
Evaluated 18/24 sequences
Evaluated 19/24 sequences
Evaluated 20/24 sequences
Evaluated 21/24 sequences
Evaluated 22/24 sequences
Evaluated 23/24 sequences
Evaluated 24/24 sequences


{'cer': 0.5741932338890514,
 'dialogue_f1': 0.717037037037037,
 'reading_order_accuracy': 0.9039800995024876,
 'speaker_accuracy': 0.49793388429752067,
 'b3_f1': 0.5757908730895792,
 'identity_switch_rate': 0.7540983606557377,
 'narration_f1': 0.0,
 'json_parse_rate': 0.9166666666666666,
 'schema_compliance_rate': 0.9166666666666666,
 'sequence_exact_match': 0.0,
 'evaluated_sequences': 24}

# RELOAD CHECKPOINT 138

Load the saved LoRA weights into the existing quantized base model. This avoids allocating a second copy of the 3B model. The original 5% grouped validation membership is intentionally retained because changing it after training would leak checkpoint training examples into evaluation.

In [5]:
CHECKPOINT_DIR = ROOT / "notebooks" / "checkpoints" / "qwen_manga_array" / "checkpoint-138"
CHECKPOINT_ADAPTER = "checkpoint_138"

if not CHECKPOINT_DIR.is_dir():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_DIR}")

# Safe to rerun: replace only the separately loaded checkpoint adapter.
if CHECKPOINT_ADAPTER in lora_model.peft_config:
    lora_model.set_adapter("default")
    lora_model.delete_adapter(CHECKPOINT_ADAPTER)

lora_model.load_adapter(
    str(CHECKPOINT_DIR),
    adapter_name=CHECKPOINT_ADAPTER,
    is_trainable=False,
)
lora_model.set_adapter(CHECKPOINT_ADAPTER)
checkpoint_model = lora_model.eval()

print(f"Loaded {CHECKPOINT_ADAPTER} from {CHECKPOINT_DIR}")

Loaded checkpoint_138 from C:\Users\ESHAAN\BYTE\Applied-Task\manga_vlm_crashcourse\notebooks\checkpoints\qwen_manga_array\checkpoint-138


In [ ]:
checkpoint_138_metrics = evaluate_model(
    model=checkpoint_model,
    processor=processor,
    dataset=val_dataset,
    collator=eval_collator,
    batch_size=1,
    max_new_tokens=1024,
    minimum_similarity=0.50,
    repetition_penalty=1.05,
    output_dir=ROOT / "outputs" / "checkpoint_138_eval",
)
checkpoint_138_metrics

In [18]:
from Scripts.evaluator import inspect_prediction

report = inspect_prediction(
    ROOT / "outputs" / "checkpoint_138_eval" / "predictions.jsonl",
    index=0,
    print_output=False
)
report.keys()
report['sequence_id']


'seq_032cbb4dc8686963'

In [19]:
report['target']

[[{'speaker': 'character1', 'text': 'I MEAN...'},
  {'speaker': 'character1', 'text': "YOU'RE THE ONLY ONE"},
  {'speaker': 'character1', 'text': "WHO'S EVER SEEN ME LIKE THIS."},
  {'speaker': 'character2', 'text': 'TH...'},
  {'speaker': 'character2', 'text': 'THE ONLY ONE...?'},
  {'speaker': 'character1', 'text': "SO I CAN'T ASK ANYONE ELSE."},
  {'speaker': 'character1', 'text': 'I...'},
  {'speaker': 'character1', 'text': 'IT HAS TO BE YOU...'},
  {'speaker': 'character2', 'text': 'UM...'},
  {'speaker': 'character2', 'text': 'ER...'},
  {'speaker': 'character2', 'text': 'UH...'},
  {'speaker': 'character2', 'text': 'WOW...'}],
 [{'speaker': 'character2', 'text': 'THEN...'},
  {'speaker': 'character2', 'text': 'JUST THIS ONCE...'},
  {'speaker': 'character1', 'text': 'YOU WILL!?'},
  {'speaker': 'NARRATION',
   'text': 'CHIKA FUJIWARA WAS THE KIND OF GIRL WHO COULD NOT SAY NO TO SOMEONE COUNTING ON HER!!'},
  {'speaker': 'character2', 'text': 'I GOT A LIVE ONE.'},
  {'speaker': '

In [20]:
report['prediction']

[[{'speaker': 'character1', 'text': 'I MEAN...'},
  {'speaker': 'character1', 'text': "YOU'RE THE ONLY ONE."},
  {'speaker': 'character1', 'text': 'WHEN EVER SEEN ME LIKE THIS.'},
  {'speaker': 'character1', 'text': 'THE ONLY ONE...'},
  {'speaker': 'character1', 'text': "SO I CAN'T ASK ANYONE ELSE."},
  {'speaker': 'character1', 'text': 'IT HAS TO BE YOU.'},
  {'speaker': 'character1', 'text': 'UM...'},
  {'speaker': 'character1', 'text': 'ER...'},
  {'speaker': 'character1', 'text': 'I-'},
  {'speaker': 'character1', 'text': 'WOW...'}],
 [{'speaker': 'character2', 'text': 'YOU WILL?'},
  {'speaker': 'character2', 'text': 'THEN...'},
  {'speaker': 'character2', 'text': 'JUST THIS ONCE...'},
  {'speaker': 'character2',
   'text': 'WHAT THE KIN-OF GIRL WHO JUST SAID NO TO SOME RING COUNCILOR ON HER!!'},
  {'speaker': 'character2', 'text': "I'M SORRY..."},
  {'speaker': 'character2', 'text': 'I GOT A LIVE ONE.'},
  {'speaker': 'character2', 'text': "LOOK, ISN'T HE CUTIE!"},
  {'speaker':

In [21]:
report['metrics']

{'cer': 0.47068676716917923,
 'cer_valid_outputs': 0.47068676716917923,
 'cer_evaluated_sequences': 1,
 'cer_including_failures_as_deletions': 0.47068676716917923,
 'dialogue_text_match_f1': 0.75,
 'dialogue_f1': 0.75,
 'matched_dialogue_order_accuracy': 0.9714285714285714,
 'reading_order_accuracy': 0.9714285714285714,
 'matched_speaker_accuracy': 0.47619047619047616,
 'speaker_accuracy': 0.47619047619047616,
 'b3_f1': 0.4637914331188154,
 'identity_switch_rate': 1.0,
 'narration_f1': 0.0,
 'json_parse_rate': 1.0,
 'schema_compliance_rate': 1.0,
 'sequence_exact_match': 0.0,
 'literal_sequence_exact_match': 0.0,
 'evaluated_sequences': 1}

In [22]:
report['generation_diagnostics']

{'generated_tokens': 502,
 'reached_eos': True,
 'hit_max_new_tokens': False,
 'suspected_repetition_loop': False}